# pregătirea datelor pentru Market Basket Analysis

Din **`c1_tranzactii_clean.csv`** calculez două tabele, folosind toate tranzacțiile curățate disponibile:

- **`basket_matrix`**: câte un rând pentru fiecare factură, 25 de produse selectate după numărul de facturi distincte și valori binare 0/1.
- **`customer_features`**: câte un rând pentru fiecare client, cu numărul comenzilor, cantitățile, valoarea totală, valoarea medie pe comandă și datele comenzilor.

Verific legătura cu tranzacțiile originale, salvez ambele tabele în **CSV și Excel** și le descarc într-o singură arhivă. Acestea devin intrările pentru analizele de asociere și clustering.

## 1. încarc tranzacțiile curățate

In [ ]:
%pip -q install openpyxl

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# în colab încarc fișierul cu tranzacțiile curățate
nume = 'c1_tranzactii_clean.csv'
if not Path(nume).exists():
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        pass
if not Path(nume).exists():
    raise FileNotFoundError('încarcă c1_tranzactii_clean.csv și rulează din nou această celulă')

tranzactii = pd.read_csv(nume, dtype={'InvoiceNo': str, 'StockCode': str, 'CustomerID': str})
tranzactii['InvoiceDate'] = pd.to_datetime(tranzactii['InvoiceDate'], errors='coerce')
coloane = ['InvoiceNo', 'StockCode', 'Quantity', 'UnitPrice', 'CustomerID', 'InvoiceDate']
if tranzactii[coloane].isna().any().any():
    raise ValueError('lipsesc valori esențiale pentru construirea tabelelor')
tranzactii['Quantity'] = pd.to_numeric(tranzactii['Quantity'], errors='raise')
tranzactii['UnitPrice'] = pd.to_numeric(tranzactii['UnitPrice'], errors='raise')
if (tranzactii['Quantity'] <= 0).any() or (tranzactii['UnitPrice'] <= 0).any():
    raise ValueError('există cantități sau prețuri nepozitive: verifică fișierul curățat')
if tranzactii['InvoiceNo'].str.startswith('C').any():
    raise ValueError('există facturi de anulare în fișierul curățat')

# folosesc aceeași valoare economică indiferent de rotunjirile coloanei TotalPrice
tranzactii['TotalPrice'] = tranzactii['Quantity'] * tranzactii['UnitPrice']
print('linii de tranzacție:', len(tranzactii))
print('facturi distincte:', tranzactii['InvoiceNo'].nunique())
print('clienți distincți:', tranzactii['CustomerID'].nunique())
display(tranzactii.head())

## 2. construiesc matricea factură–produs

In [ ]:
# aleg 25 de produse după numărul de facturi în care apar, nu după cantitatea vândută
frecventa = (tranzactii.groupby('StockCode')['InvoiceNo']
             .nunique().sort_values(ascending=False))
produse_25 = frecventa.head(25).index.tolist()

# verific pentru fiecare factură dacă produsul apare cel puțin o dată
prezenta = (tranzactii.loc[tranzactii['StockCode'].isin(produse_25),
                          ['InvoiceNo', 'StockCode']]
            .drop_duplicates().assign(Prezent=1))
matrice = prezenta.pivot(index='InvoiceNo', columns='StockCode', values='Prezent')

# păstrez toate facturile, inclusiv pe cele fără produsele selectate
facturi = pd.Index(tranzactii['InvoiceNo'].drop_duplicates(), name='InvoiceNo')
matrice = matrice.reindex(index=facturi, columns=produse_25).fillna(0).astype('int8')
basket_matrix = matrice.reset_index()

assert len(basket_matrix) == tranzactii['InvoiceNo'].nunique()
assert basket_matrix['InvoiceNo'].is_unique
assert matrice.isin([0, 1]).all().all()
assert list(matrice.columns) == produse_25
print('dimensiunea matricei:', basket_matrix.shape)
print('facturi fără niciun produs din top 25:', int((matrice.sum(axis=1) == 0).sum()))
print('formula: 1 = produs prezent pe factură; 0 = produs absent')
display(basket_matrix.head())

## 3. calculez indicatorii clienților

In [ ]:
# o comandă este identificată prin numărul distinct al facturii
clienti = tranzactii.groupby('CustomerID').agg(
    NrComenzi=('InvoiceNo', 'nunique'),
    NrProduse=('Quantity', 'sum'),
    ValoareTotala=('TotalPrice', 'sum'),
    NrLinii=('InvoiceNo', 'size'),
    NrProduseDistincte=('StockCode', 'nunique'),
    PrimaComanda=('InvoiceDate', 'min'),
    UltimaComanda=('InvoiceDate', 'max')
).reset_index()

# media pe comandă reprezintă suma cumpărăturilor împărțită la numărul comenzilor
clienti['MediePeComanda'] = clienti['ValoareTotala'] / clienti['NrComenzi']

# recency folosește aceeași dată de referință pentru toți clienții
ultima_zi = tranzactii['InvoiceDate'].max().normalize()
data_referinta = ultima_zi + pd.Timedelta(days=1)
clienti['RecencyZile'] = (data_referinta - clienti['UltimaComanda'].dt.normalize()).dt.days
customer_features = clienti[['CustomerID', 'NrComenzi', 'NrProduse', 'ValoareTotala',
                             'MediePeComanda', 'NrLinii', 'NrProduseDistincte',
                             'PrimaComanda', 'UltimaComanda', 'RecencyZile']]

assert customer_features['CustomerID'].is_unique
assert len(customer_features) == tranzactii['CustomerID'].nunique()
assert np.isclose(customer_features['ValoareTotala'].sum(), tranzactii['TotalPrice'].sum())
assert (customer_features['RecencyZile'] >= 1).all()
print('clienți:', len(customer_features))
print('data de referință pentru recency:', data_referinta.date())
display(customer_features.head())

## 4. verific rezultatele și le salvez în CSV și Excel

In [ ]:
iesiri = Path('date_generate')
iesiri.mkdir(exist_ok=True)
basket_matrix.to_csv(iesiri / 'basket_matrix.csv', index=False)
basket_matrix.to_excel(iesiri / 'basket_matrix.xlsx', index=False, sheet_name='BasketMatrix')
customer_features.to_csv(iesiri / 'customer_features.csv', index=False, date_format='%Y-%m-%d %H:%M:%S')
customer_features.to_excel(iesiri / 'customer_features.xlsx', index=False, sheet_name='CustomerFeatures')

# verific fișierele generate și valorile calculate
verif_basket = pd.read_excel(iesiri / 'basket_matrix.xlsx', dtype={'InvoiceNo': str})
verif_clienti = pd.read_excel(iesiri / 'customer_features.xlsx', dtype={'CustomerID': str})
assert verif_basket.shape == basket_matrix.shape
assert verif_clienti.shape == customer_features.shape
assert np.isclose(verif_clienti['ValoareTotala'].sum(), tranzactii['TotalPrice'].sum())

arhiva = Path('basket_si_customer_features.zip')
with ZipFile(arhiva, 'w') as z:
    for f in sorted(iesiri.iterdir()):
        z.write(f, arcname=f.name)
print('am generat:', ', '.join(f.name for f in sorted(iesiri.iterdir())))
print('arhiva:', arhiva)
try:
    from google.colab import files
    files.download(str(arhiva))
except ImportError:
    print('arhiva este disponibilă în directorul de lucru')